# Get MS/MS embeddings

[Open in Colab](https://colab.research.google.com/github/Dsadd4/UltraMS/blob/main/cookbook/tutorials/embed.ipynb)

Read a real MGF file and obtain one UltraMS embedding per spectrum. The input file is the [five-spectrum DreaMS example](../../examples/data/NOTICE.md). Run cells from top to bottom.

## Install

In [ ]:
%pip install -q ultrams


## Read spectra

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import torch
from ultrams import UltraMS, read_spectra

sample = Path("examples/data/example_5_spectra.mgf")
if not sample.exists():
    sample = Path("example_5_spectra.mgf")
    if not sample.exists():
        urlretrieve(
            "https://raw.githubusercontent.com/Dsadd4/UltraMS/main/examples/data/example_5_spectra.mgf",
            sample,
        )
spectra = list(read_spectra(sample))
assert len(spectra) == 5
print(f"Loaded {len(spectra)} MS/MS spectra")

## Encode in batches

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = UltraMS.from_pretrained("unsupervised", device=device)
embeddings = model.encode_batch(spectra, batch_size=5)
assert embeddings.shape == (len(spectra), model.embedding_dim)
print("Embedding matrix:", embeddings.shape)
print("First spectrum:", spectra[0]["id"])

## Save embeddings

In [ ]:
import numpy as np

output = Path("ultrams_embeddings.npz")
np.savez_compressed(output, ids=[spectrum["id"] for spectrum in spectra], embeddings=embeddings)
print(output.resolve())